# 🎙️ Realtime Voice AI Agent with RAG — Colab Showcase

Full-stack demo maintaining the **exact project structure** of the production deployment,
replacing AWS with Colab-friendly equivalents.

---

## 🔑 Required Colab Secrets

| Secret Name | Description |
|-------------|-------------|
| `GROQ_API_KEY` | Free at [console.groq.com](https://console.groq.com) |
| `MONGO_URL` | MongoDB Atlas connection string |
| `DEEPGRAM_API_KEY` | Free at [console.deepgram.com](https://console.deepgram.com) |
| `ELEVENLABS_API_KEY` | Optional — gTTS fallback used if absent |

> No Google or AWS keys needed. Embeddings use `sentence-transformers/all-MiniLM-L6-v2` — free and local.

---

## 📁 Project Structure
```
backend/
  app/
    __init__.py  config.py  database.py  bot.py
    models/      document.py  equipment.py  rag.py
    routers/     equipment.py  stream.py
    services/    embeddings.py  rag.py  text_extraction.py
  main.py
.github/workflows/deploy.yml   (CI/CD - Docker Hub, no AWS)
```


## 📦 Section 1 — Install Dependencies

In [2]:
# All packages — LangChain >=1.2 compatible imports used throughout
!pip install -q \
    fastapi "uvicorn[standard]" motor pymongo \
    "langchain>=0.2.0" "langchain-community>=0.2.0" "langchain-text-splitters>=0.2.0" \
    sentence-transformers \
    pypdf python-docx python-multipart \
    pydantic "pydantic-settings" python-dotenv \
    loguru httpx requests groq \
    gtts nest_asyncio aiofiles ipywidgets
print("✅ All packages installed")

✅ All packages installed


## 🗂️ Section 2 — Create Project Directory Structure

In [3]:
import os, sys

dirs = [
    "backend/app/models",
    "backend/app/routers",
    "backend/app/services",
    ".github/workflows",
    "scripts",
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

# Add backend to sys.path so absolute imports match production
if os.path.abspath("backend") not in sys.path:
    sys.path.insert(0, os.path.abspath("backend"))

print("✅ Project directories created")
for root, subdirs, files in os.walk("backend"):
    level = root.replace("backend", "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for f in sorted(files):
        print("  " * (level + 1) + f)

✅ Project directories created
backend/
  app/
    services/
    routers/
    models/


## ⚙️ Section 3 — `backend/app/config.py`

In [4]:
CONFIG_PY = 'from pydantic_settings import BaseSettings\nfrom typing import Optional\n\nclass Settings(BaseSettings):\n    MONGO_URL: str\n    DB_NAME: str = "live_db"\n    GROQ_API_KEY: str\n    DEEPGRAM_API_KEY: str = ""\n    ELEVENLABS_API_KEY: str = ""\n    ELEVENLABS_VOICE_ID: str = "pNInz6obpgDQGcFmaJgB"\n    # llama-3.1-8b-instant: fast, 131k ctx, free tier friendly\n    # llama-3.3-70b-versatile: higher quality, fits token limits, also free\n    GROQ_MODEL: str = "llama-3.1-8b-instant"\n    GROQ_BASE_URL: str = "https://api.groq.com/openai/v1"\n    # Free local HuggingFace embeddings - no API key needed\n    EMBEDDING_MODEL: str = "sentence-transformers/all-MiniLM-L6-v2"\n    EMBEDDING_DIMENSIONS: int = 384\n    CHUNK_SIZE: int = 800\n    CHUNK_OVERLAP: int = 150\n    VECTOR_INDEX_NAME: str = "vector_index"\n    DOCUMENT_CHUNKS_COLLECTION: str = "document_chunks"\n    TENANT_ID: str = "mvp_tenant"\n    USER_ID: str = "mvp_user"\n\n    class Config:\n        env_file = ".env"\n        case_sensitive = True\n\nsettings = Settings()\n'

with open("backend/app/config.py", "w") as f:
    f.write(CONFIG_PY)
print("✅ backend/app/config.py written")

✅ backend/app/config.py written


In [12]:
import os
from google.colab import userdata

def load_secret(name, default=None, required=False):
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f"  ✅ {name} loaded")
            return val
    except Exception:
        pass
    if required:
        raise EnvironmentError(f"❌ Required secret {name} not found in Colab Secrets!")
    if default is not None:
        os.environ[name] = default
        print(f"  ⚠️  {name} not found — using default")
    return default

print("Loading secrets...")
load_secret("GROQ_API_KEY",       required=True)
load_secret("MONGO_DB_URL",       required=True) # Load the user's secret
load_secret("DEEPGRAM_API_KEY",   default="")
load_secret("ELEVENLABS_API_KEY", default="")

# Map MONGO_DB_URL to MONGO_URL for the app's config
if "MONGO_DB_URL" in os.environ:
    os.environ["MONGO_URL"] = os.environ["MONGO_DB_URL"]

from app.config import settings
print(f"\n🔧 Config loaded:")
print(f"   GROQ_MODEL      = {settings.GROQ_MODEL}")
print(f"   EMBEDDING_MODEL = {settings.EMBEDDING_MODEL}")
print(f"   DB_NAME         = {settings.DB_NAME}")
print(f"   TENANT_ID       = {settings.TENANT_ID}")

Loading secrets...
  ✅ GROQ_API_KEY loaded
  ✅ MONGO_DB_URL loaded
  ✅ DEEPGRAM_API_KEY loaded
  ✅ ELEVENLABS_API_KEY loaded

🔧 Config loaded:
   GROQ_MODEL      = llama-3.1-8b-instant
   EMBEDDING_MODEL = sentence-transformers/all-MiniLM-L6-v2
   DB_NAME         = live_db
   TENANT_ID       = mvp_tenant


## 🧩 Section 4 — Pydantic Models (`backend/app/models/`)

In [13]:
RAG_MODEL = 'from pydantic import BaseModel, Field\nfrom typing import Optional, List\n\nclass ChunkContent(BaseModel):\n    text: str = Field(..., description="The actual text content of the chunk")\n    file_name: Optional[str] = None\n    score: Optional[float] = None\n\nclass ChunkMetadata(BaseModel):\n    chunk_id: str\n    document_id: str\n    equipment_id: str\n    tenant_id: Optional[str] = None\n    chunk_index: int\n    score: float\n    file_name: str\n\nclass RetrievalMetadata(BaseModel):\n    query: str\n    k: int\n    chunks_retrieved: int\n    equipment_id: Optional[str] = None\n    tenant_id: Optional[str] = None\n    chunks: List[ChunkMetadata] = []\n\nclass RetrievalResult(BaseModel):\n    data: List[ChunkContent]\n    metadata: RetrievalMetadata\n'
DOC_MODEL = 'from pydantic import BaseModel\nfrom typing import Optional\nfrom datetime import datetime\n\nclass Document(BaseModel):\n    equipment_id: str\n    tenant_id: str\n    file_name: str\n    content_type: str\n    size: int\n    storage_key: str\n    uploaded_by: str\n    description: Optional[str] = None\n    embedding_status: str = "pending"\n    embedding_error: Optional[dict] = None\n    created_at: Optional[datetime] = None\n    updated_at: Optional[datetime] = None\n\n    class Config:\n        arbitrary_types_allowed = True\n        json_encoders = {datetime: lambda v: v.isoformat()}\n'
EQUIP_MODEL = 'from pydantic import BaseModel, Field, ConfigDict\nfrom typing import Optional\nfrom datetime import datetime\n\nclass Equipment(BaseModel):\n    id: Optional[str] = Field(None, alias="_id", serialization_alias="_id")\n    name: str\n    description: str\n    tenant_id: str\n    is_active: bool = True\n    created_at: Optional[datetime] = None\n    updated_at: Optional[datetime] = None\n\n    model_config = ConfigDict(\n        populate_by_name=True,\n        arbitrary_types_allowed=True,\n        json_encoders={datetime: lambda v: v.isoformat()}\n    )\n'

for path, content in [
    ("backend/app/__init__.py", ""),
    ("backend/app/models/__init__.py", ""),
    ("backend/app/models/rag.py", RAG_MODEL),
    ("backend/app/models/document.py", DOC_MODEL),
    ("backend/app/models/equipment.py", EQUIP_MODEL),
]:
    with open(path, "w") as f:
        f.write(content)
print("✅ All models written")

✅ All models written


## 🗄️ Section 5 — MongoDB Connection (`backend/app/database.py`)

In [14]:
DB_PY = 'from motor.motor_asyncio import AsyncIOMotorClient, AsyncIOMotorDatabase\nfrom app.config import settings\nfrom loguru import logger\n\nclient: AsyncIOMotorClient = None\ndatabase: AsyncIOMotorDatabase = None\n\nasync def connect_to_mongo():\n    global client, database\n    try:\n        client = AsyncIOMotorClient(\n            settings.MONGO_URL,\n            serverSelectionTimeoutMS=30000,\n            connectTimeoutMS=30000,\n            socketTimeoutMS=30000,\n        )\n        database = client[settings.DB_NAME]\n        await client.admin.command("ping")\n        logger.info(f"✅ Connected to MongoDB: {settings.DB_NAME}")\n    except Exception as e:\n        logger.error(f"❌ Failed to connect to MongoDB: {str(e)}")\n        raise\n\nasync def close_mongo_connection():\n    global client\n    if client:\n        client.close()\n        logger.info("✅ MongoDB connection closed")\n\ndef get_database() -> AsyncIOMotorDatabase:\n    return database\n'

with open("backend/app/database.py", "w") as f:
    f.write(DB_PY)
print("✅ backend/app/database.py written")

✅ backend/app/database.py written


In [15]:
import asyncio, nest_asyncio
nest_asyncio.apply()
from app.database import connect_to_mongo, get_database

async def test_connection():
    await connect_to_mongo()
    db = get_database()
    collections = await db.list_collection_names()
    print(f"✅ Connected to: {db.name}")
    print(f"   Collections: {collections if collections else ["(empty - fresh DB)"]}")

asyncio.get_event_loop().run_until_complete(test_connection())

2026-05-06 10:23:13.574 | INFO     | app.database:connect_to_mongo:19 - ✅ Connected to MongoDB: live_db


✅ Connected to: live_db
   Collections: ['(empty - fresh DB)']


## 🛠️ Section 6 — Services

### 6a — Embeddings Service (`backend/app/services/embeddings.py`)

> **Free, local, no API key.** Uses `sentence-transformers/all-MiniLM-L6-v2` (384 dims).
> Replaces paid Google `text-embedding-004` from production.
> LangChain `>=1.2` imports: `langchain_community`, `langchain_text_splitters`.


In [16]:
EMBEDDINGS_PY = 'from typing import List\nfrom loguru import logger\n# LangChain >=1.2 imports\nfrom langchain_text_splitters import RecursiveCharacterTextSplitter\nfrom langchain_community.embeddings import HuggingFaceEmbeddings\nfrom app.config import settings\n\nclass EmbeddingService:\n    """\n    FREE embedding service using sentence-transformers/all-MiniLM-L6-v2.\n    No API key required. 384-dimensional vectors. Fast CPU inference.\n    Drop-in replacement for GoogleGenerativeAIEmbeddings.\n    """\n    def __init__(self):\n        logger.info(f"Loading embedding model: {settings.EMBEDDING_MODEL}")\n        self.embeddings = HuggingFaceEmbeddings(\n            model_name=settings.EMBEDDING_MODEL,\n            model_kwargs={"device": "cpu"},\n            encode_kwargs={"normalize_embeddings": True},\n        )\n        self.text_splitter = RecursiveCharacterTextSplitter(\n            chunk_size=settings.CHUNK_SIZE,\n            chunk_overlap=settings.CHUNK_OVERLAP,\n            length_function=len,\n            is_separator_regex=False,\n        )\n        logger.info("✅ EmbeddingService ready")\n\n    def split_text(self, text: str) -> List[str]:\n        if not text or not text.strip():\n            return []\n        return self.text_splitter.split_text(text)\n\n    def embed_text(self, text: str) -> List[float]:\n        if not text or not text.strip():\n            raise ValueError("Cannot embed empty text")\n        return self.embeddings.embed_query(text)\n\n    def embed_texts(self, texts: List[str]) -> List[List[float]]:\n        if not texts:\n            return []\n        valid = [t for t in texts if t and t.strip()]\n        return self.embeddings.embed_documents(valid) if valid else []\n'

with open("backend/app/services/__init__.py", "w") as f:
    f.write("")
with open("backend/app/services/embeddings.py", "w") as f:
    f.write(EMBEDDINGS_PY)
print("✅ backend/app/services/embeddings.py written")

✅ backend/app/services/embeddings.py written


In [17]:
from app.services.embeddings import EmbeddingService

print("Loading embedding model (downloads ~90MB on first run)...")
emb_svc = EmbeddingService()
vec = emb_svc.embed_text("How do I reset the hydraulic pressure?")
print(f"✅ Embedding dimensions: {len(vec)}")
print(f"   First 5 values: {[round(v, 4) for v in vec[:5]]}")
chunks = emb_svc.split_text("Test sentence. " * 60)
print(f"✅ Split test text into {len(chunks)} chunks")

2026-05-06 10:24:01.139 | INFO     | app.services.embeddings:__init__:15 - Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading embedding model (downloads ~90MB on first run)...


/content/backend/app/services/embeddings.py:16: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-05-06 10:24:07.815 | INFO     | app.services.embeddings:__init__:27 - ✅ EmbeddingService ready


✅ Embedding dimensions: 384
   First 5 values: [-0.0379, -0.0222, 0.021, 0.0018, -0.084]
✅ Split test text into 2 chunks


### 6b — Text Extraction Service (`backend/app/services/text_extraction.py`)

In [19]:
TEXT_EXT_PY = 'import os\nfrom pypdf import PdfReader\nfrom docx import Document as DocxDocument\n\nclass TextExtractionService:\n\n    SUPPORTED_FORMATS = {\n        "text/plain": ["txt", "md"],\n        "application/pdf": ["pdf"],\n        "application/vnd.openxmlformats-officedocument.wordprocessingml.document": ["docx"],\n    }\n\n    def extract_text(self, file_path: str, content_type: str) -> str:\n        if not os.path.exists(file_path):\n            raise FileNotFoundError(f"File not found: {file_path}")\n        ext = self._get_extension(file_path)\n        if content_type == "text/plain" or ext in ["txt", "md"]:\n            return self._extract_text_file(file_path)\n        elif content_type == "application/pdf" or ext == "pdf":\n            return self._extract_pdf(file_path)\n        elif "wordprocessingml" in content_type or ext == "docx":\n            return self._extract_docx(file_path)\n        else:\n            raise ValueError(f"Unsupported file format: {content_type} ({ext})")\n\n    def _extract_text_file(self, file_path: str) -> str:\n        try:\n            with open(file_path, "r", encoding="utf-8") as f:\n                return f.read().strip()\n        except UnicodeDecodeError:\n            with open(file_path, "r", encoding="latin-1") as f:\n                return f.read().strip()\n\n    def _extract_pdf(self, file_path: str) -> str:\n        reader = PdfReader(file_path)\n        parts = [p.extract_text() for p in reader.pages if p.extract_text()]\n        return "\\n\\n".join(parts).strip()\n\n    def _extract_docx(self, file_path: str) -> str:\n        doc = DocxDocument(file_path)\n        parts = [p.text for p in doc.paragraphs if p.text.strip()]\n        for table in doc.tables:\n            for row in table.rows:\n                for cell in row.cells:\n                    if cell.text.strip():\n                        parts.append(cell.text)\n        return "\\n\\n".join(parts).strip()\n\n    def is_supported(self, content_type: str, file_path: str) -> bool:\n        ext = self._get_extension(file_path)\n        if content_type in self.SUPPORTED_FORMATS:\n            return True\n        return any(ext in exts for exts in self.SUPPORTED_FORMATS.values())\n\n    def _get_extension(self, file_path: str) -> str:\n        return os.path.splitext(file_path)[1].lstrip(".").lower()\n'

with open("backend/app/services/text_extraction.py", "w") as f:
    f.write(TEXT_EXT_PY)
print("✅ backend/app/services/text_extraction.py written")

✅ backend/app/services/text_extraction.py written


### 6c — RAG Service (`backend/app/services/rag.py`)

In [20]:
RAG_SVC_PY = 'from typing import Any, List, Optional\nimport uuid\nfrom bson import ObjectId\nfrom loguru import logger\nfrom app.database import get_database\nfrom app.services.embeddings import EmbeddingService\nfrom app.config import settings\nfrom app.models.rag import ChunkContent, ChunkMetadata, RetrievalMetadata, RetrievalResult\n\nembeddings_service = EmbeddingService()\n\nclass RAGService:\n    def __init__(self, index_name: str = None):\n        self.index_name = index_name or settings.VECTOR_INDEX_NAME\n\n    async def retrieve(\n        self,\n        query: str,\n        k: int = 5,\n        equipment_id: Optional[str] = None,\n        tenant_id: Optional[str] = None,\n        extra_filters: Optional[dict] = None,\n    ) -> RetrievalResult:\n        """\n        Retrieve top-k semantically similar chunks.\n        Tries MongoDB Atlas Vector Search first.\n        Falls back to in-memory cosine similarity when Atlas index is not configured.\n        """\n        db = get_database()\n        collection = db[settings.DOCUMENT_CHUNKS_COLLECTION]\n\n        logger.info(f"Retrieval query: {query[:60]} (k={k})")\n        query_embedding = embeddings_service.embed_text(query)\n\n        filters: dict = {"is_disabled": {"$ne": True}}\n        if equipment_id:\n            try:\n                filters["equipment_id"] = ObjectId(equipment_id)\n            except Exception:\n                logger.warning(f"Invalid equipment_id; skipping filter")\n        if tenant_id:\n            filters["tenant_id"] = tenant_id\n        if extra_filters:\n            filters.update(extra_filters)\n\n        try:\n            pipeline = [\n                {\n                    "$vectorSearch": {\n                        "index": self.index_name,\n                        "path": "embedding",\n                        "queryVector": query_embedding,\n                        "numCandidates": k * 5,\n                        "limit": k,\n                        **({"filter": filters} if filters else {}),\n                    }\n                },\n                {\n                    "$project": {\n                        "_id": 1, "chunk_id": 1, "document_id": 1,\n                        "file_name": 1, "text": 1, "chunk_index": 1,\n                        "equipment_id": 1, "tenant_id": 1,\n                        "score": {"$meta": "vectorSearchScore"},\n                    }\n                },\n            ]\n            cursor = collection.aggregate(pipeline)\n            results = await cursor.to_list(length=k)\n            if not results:\n                raise ValueError("Atlas returned 0 results")\n            logger.info(f"Atlas vector search: {len(results)} results")\n        except Exception as e:\n            logger.warning(f"Atlas Vector Search unavailable ({e}). Using cosine fallback.")\n            results = await self._cosine_fallback(collection, query_embedding, filters, k)\n\n        chunk_data, chunk_meta = [], []\n        for res in results:\n            chunk_data.append(ChunkContent(\n                text=res.get("text", ""),\n                file_name=res.get("file_name"),\n                score=res.get("score"),\n            ))\n            chunk_meta.append(ChunkMetadata(\n                chunk_id=res.get("chunk_id", ""),\n                document_id=str(res.get("document_id", "")),\n                equipment_id=str(res.get("equipment_id", "")),\n                tenant_id=res.get("tenant_id"),\n                chunk_index=res.get("chunk_index", 0),\n                score=float(res.get("score", 0.0)),\n                file_name=res.get("file_name", ""),\n            ))\n\n        return RetrievalResult(\n            data=chunk_data,\n            metadata=RetrievalMetadata(\n                query=query, k=k,\n                chunks_retrieved=len(chunk_data),\n                equipment_id=equipment_id, tenant_id=tenant_id,\n                chunks=chunk_meta,\n            ),\n        )\n\n    async def _cosine_fallback(self, collection, query_embedding, filters, k):\n        """In-memory cosine similarity when Atlas Vector Search index is not set up."""\n        import numpy as np\n        docs = await collection.find(filters).to_list(length=1000)\n        if not docs:\n            return []\n        qv = np.array(query_embedding)\n        scored = []\n        for doc in docs:\n            ev = doc.get("embedding")\n            if not ev:\n                continue\n            ev = np.array(ev)\n            norm = np.linalg.norm(qv) * np.linalg.norm(ev)\n            doc["score"] = float(np.dot(qv, ev) / norm) if norm > 0 else 0.0\n            scored.append(doc)\n        scored.sort(key=lambda d: d["score"], reverse=True)\n        return scored[:k]\n'

with open("backend/app/services/rag.py", "w") as f:
    f.write(RAG_SVC_PY)
print("✅ backend/app/services/rag.py written (with cosine fallback)")

✅ backend/app/services/rag.py written (with cosine fallback)


## 🔌 Section 7 — FastAPI Routers

### 7a — Equipment Router (`backend/app/routers/equipment.py`)

In [21]:
EQUIP_ROUTER = 'import os, uuid, tempfile\nfrom datetime import datetime\nfrom typing import List, Optional\nfrom fastapi import APIRouter, HTTPException, status, UploadFile, File, Form\nfrom loguru import logger\nfrom bson import ObjectId\nfrom app.services.text_extraction import TextExtractionService\nfrom app.services.embeddings import EmbeddingService\nfrom app.database import get_database\nfrom app.models.equipment import Equipment\nfrom app.config import settings\n\nrouter = APIRouter()\n\ndef _serialize(doc: dict) -> dict:\n    out = dict(doc)\n    for key in ("_id", "equipment_id", "document_id"):\n        if key in out and isinstance(out[key], ObjectId):\n            out[key] = str(out[key])\n    for key in ("created_at", "updated_at"):\n        if key in out and isinstance(out[key], datetime):\n            out[key] = out[key].isoformat()\n    return out\n\n@router.post("/", response_model=Equipment, status_code=status.HTTP_201_CREATED)\nasync def create_equipment(equipment: Equipment):\n    db = get_database()\n    existing = await db.equipment.find_one({"name": equipment.name, "tenant_id": equipment.tenant_id})\n    if existing:\n        raise HTTPException(status_code=409, detail="Equipment with this name already exists")\n    now = datetime.utcnow()\n    eq_dict = equipment.model_dump(exclude={"id"}, exclude_none=True)\n    eq_dict.update({"created_at": now, "updated_at": now})\n    result = await db.equipment.insert_one(eq_dict)\n    resp = equipment.model_dump(exclude={"id"}, exclude_none=True)\n    resp["_id"] = str(result.inserted_id)\n    return Equipment(**resp)\n\n@router.get("/", response_model=List[Equipment])\nasync def get_equipment():\n    db = get_database()\n    items = await db.equipment.find({}).to_list(length=None)\n    return [Equipment(**_serialize(i)) for i in items]\n\n@router.get("/{equipment_id}", response_model=Equipment)\nasync def get_one_equipment(equipment_id: str):\n    db = get_database()\n    item = await db.equipment.find_one({"_id": ObjectId(equipment_id)})\n    if not item:\n        raise HTTPException(status_code=404, detail="Equipment not found")\n    return Equipment(**_serialize(item))\n\n@router.post("/{equipment_id}/documents", status_code=201)\nasync def upload_equipment_documents(\n    equipment_id: str,\n    files: List[UploadFile] = File(...),\n    description: Optional[str] = Form(None),\n):\n    db = get_database()\n    equipment = await db.equipment.find_one({"_id": ObjectId(equipment_id)})\n    if not equipment:\n        raise HTTPException(status_code=404, detail="Equipment not found")\n    text_extractor = TextExtractionService()\n    embedding_service = EmbeddingService()\n    tenant_id = settings.TENANT_ID\n    created_docs = []\n    for file in files:\n        try:\n            data = await file.read()\n            original_name = file.filename or "upload.bin"\n            content_type = file.content_type or "application/octet-stream"\n            logger.info(f"Processing: {original_name} ({len(data)} bytes)")\n            if not text_extractor.is_supported(content_type, original_name):\n                logger.warning(f"Unsupported format: {content_type}")\n                continue\n            temp_path = None\n            try:\n                _, ext = os.path.splitext(original_name)\n                with tempfile.NamedTemporaryFile(delete=False, suffix=ext) as tmp:\n                    tmp.write(data)\n                    temp_path = tmp.name\n                extracted_text = text_extractor.extract_text(temp_path, content_type)\n                if not extracted_text or not extracted_text.strip():\n                    logger.warning(f"No text extracted from {original_name}")\n                    continue\n                chunks = embedding_service.split_text(extracted_text)\n                logger.info(f"Split into {len(chunks)} chunks")\n                if not chunks:\n                    continue\n                storage_key = f"{tenant_id}/equipment/{equipment_id}/{uuid.uuid4().hex}-{original_name}"\n                now = datetime.utcnow()\n                doc_dict = {\n                    "equipment_id": ObjectId(equipment_id),\n                    "tenant_id": tenant_id,\n                    "file_name": original_name,\n                    "content_type": content_type,\n                    "size": len(data),\n                    "storage_key": storage_key,\n                    "uploaded_by": settings.USER_ID,\n                    "description": description,\n                    "embedding_status": "processing",\n                    "created_at": now,\n                    "updated_at": now,\n                }\n                doc_result = await db.documents_metadata.insert_one(doc_dict)\n                document_id = doc_result.inserted_id\n                chunk_docs = []\n                for idx, chunk_text in enumerate(chunks):\n                    try:\n                        vec = embedding_service.embed_text(chunk_text)\n                        chunk_docs.append({\n                            "document_id": document_id,\n                            "equipment_id": ObjectId(equipment_id),\n                            "tenant_id": tenant_id,\n                            "file_name": original_name,\n                            "chunk_id": str(uuid.uuid4()),\n                            "chunk_index": idx,\n                            "text": chunk_text,\n                            "embedding": vec,\n                            "is_disabled": False,\n                        })\n                    except Exception as e:\n                        logger.warning(f"Chunk {idx} embedding failed: {e}")\n                if chunk_docs:\n                    await db[settings.DOCUMENT_CHUNKS_COLLECTION].insert_many(chunk_docs)\n                    await db.documents_metadata.update_one(\n                        {"_id": document_id},\n                        {"$set": {"embedding_status": "completed", "updated_at": datetime.utcnow()}}\n                    )\n                    logger.success(f"✅ {original_name} - {len(chunk_docs)} chunks stored")\n                    doc_dict["_id"] = str(document_id)\n                    doc_dict["equipment_id"] = str(doc_dict["equipment_id"])\n                    doc_dict["created_at"] = doc_dict["created_at"].isoformat()\n                    doc_dict["updated_at"] = doc_dict["updated_at"].isoformat()\n                    created_docs.append(doc_dict)\n                else:\n                    await db.documents_metadata.update_one(\n                        {"_id": document_id},\n                        {"$set": {"embedding_status": "failed", "updated_at": datetime.utcnow()}}\n                    )\n            finally:\n                if temp_path and os.path.exists(temp_path):\n                    try:\n                        os.remove(temp_path)\n                    except Exception:\n                        pass\n        except Exception as e:\n            logger.error(f"Error processing {file.filename}: {e}", exc_info=True)\n    return {"documents": created_docs, "count": len(created_docs)}\n\n@router.get("/{equipment_id}/documents")\nasync def list_equipment_documents(equipment_id: str):\n    db = get_database()\n    if not await db.equipment.find_one({"_id": ObjectId(equipment_id)}):\n        raise HTTPException(status_code=404, detail="Equipment not found")\n    docs = await db.documents_metadata.find({\n        "equipment_id": ObjectId(equipment_id),\n        "is_disabled": {"$ne": True},\n    }).to_list(length=1000)\n    return {"documents": [_serialize(d) for d in docs], "count": len(docs)}\n'

with open("backend/app/routers/__init__.py", "w") as f:
    f.write("")
with open("backend/app/routers/equipment.py", "w") as f:
    f.write(EQUIP_ROUTER)
print("✅ backend/app/routers/equipment.py written")

✅ backend/app/routers/equipment.py written


### 7b — Stream Router (`backend/app/routers/stream.py`)

In [22]:
STREAM_ROUTER = 'from typing import Dict, Any\nfrom fastapi import APIRouter, Request, HTTPException\nfrom loguru import logger\nfrom bson import ObjectId\nfrom app.database import get_database\n\nrouter = APIRouter()\n\n@router.post("/connect")\nasync def bot_connect(request: Request) -> Dict[str, Any]:\n    """\n    Returns WebSocket URL for the voice bot.\n    Production: AWS ECS/ALB endpoint.\n    Colab: localhost ws:// URL for demonstration.\n    """\n    try:\n        body: Dict[str, Any] = await request.json()\n    except Exception as e:\n        raise HTTPException(status_code=400, detail=f"Invalid JSON: {e}")\n    equipment_id: str = body.get("equipment_id", "")\n    if not equipment_id:\n        raise HTTPException(status_code=400, detail="equipment_id is required")\n    db = get_database()\n    try:\n        equipment = await db.equipment.find_one({"_id": ObjectId(equipment_id)})\n    except Exception:\n        raise HTTPException(status_code=400, detail="Invalid equipment_id format")\n    if not equipment:\n        raise HTTPException(status_code=404, detail=f"Equipment {equipment_id} not found")\n    ws_url = f"ws://localhost:8000/api/v1/stream/ws/{equipment_id}"\n    logger.info(f"Generated WS URL: {ws_url}")\n    return {"ws_url": ws_url}\n'

with open("backend/app/routers/stream.py", "w") as f:
    f.write(STREAM_ROUTER)
print("✅ backend/app/routers/stream.py written")

✅ backend/app/routers/stream.py written


## 🤖 Section 8 — AI Bot Core (`backend/app/bot.py`)

> Production: runs inside a Pipecat WebSocket pipeline (VAD -> STT -> LLM -> TTS).
> Colab: `run_rag_query()` exposes the identical Groq prompt, tool schema, and RAG retrieval as a callable.


In [23]:
BOT_PY = '"""\nCore AI bot logic - Groq LLM + RAG tool function call.\nIn production this is wrapped in a Pipecat WebSocket pipeline.\nIn Colab we expose run_rag_query() for direct demonstration.\n"""\nimport json\nfrom typing import Dict, Any, List, Optional\nfrom loguru import logger\nfrom groq import AsyncGroq\nfrom app.services.rag import RAGService\nfrom app.config import settings\n\n# System prompt - identical to production bot.py\nSYSTEM_PROMPT = """\nYou are an AI assistant supporting a human call-center agent.\n\nGoal:\nProvide the human agent with fast, efficient guidance suitable for real-time conversation.\nSpeak in natural, concise sentences. Do NOT output JSON.\n\nBehavioral rules:\n- Implement a natural, helpful, and professional tone.\n- Keep responses brief and to the point (optimized for speech).\n- Do not read out chunk IDs or metadata unless explicitly asked.\n\nKnowledge base rules:\n- When the customer asks a question or seeks information, call the search_knowledge_base tool.\n- Use ONLY facts returned from the knowledge base to answer questions.\n- If the knowledge base lacks the answer, briefly suggest that the agent apologize and ask for clarification.\n- NEVER invent or guess information.\n\nContent generation:\n- Your output will be converted to speech, so avoid special characters or complex formatting.\n- Directly address the agent with the guidance or answer.\n\nAnswer in one or two sentences and under 30 words.\nAnswer prices in integers and do not include any decimal places.\n"""\n\n# Tool schema - mirrors production FunctionSchema\nSEARCH_TOOL = {\n    "type": "function",\n    "function": {\n        "name": "search_knowledge_base",\n        "description": "Search the knowledge base for relevant information about equipment",\n        "parameters": {\n            "type": "object",\n            "properties": {\n                "query": {"type": "string", "description": "The search query"}\n            },\n            "required": ["query"],\n        },\n    },\n}\n\nasync def run_rag_query(\n    user_message: str,\n    equipment_id: Optional[str] = None,\n    tenant_id: Optional[str] = None,\n    conversation_history: Optional[List[Dict]] = None,\n) -> Dict[str, Any]:\n    """\n    Run one turn of the RAG-augmented LLM.\n    Mirrors production pipeline: STT output -> LLM -> tool_call -> RAG -> LLM -> TTS input.\n    Returns: answer (str), chunks (list), tool_called (bool)\n    """\n    client = AsyncGroq(api_key=settings.GROQ_API_KEY)\n    rag_service = RAGService()\n\n    messages = [{"role": "system", "content": SYSTEM_PROMPT}]\n    if conversation_history:\n        messages.extend(conversation_history)\n    messages.append({"role": "user", "content": user_message})\n\n    retrieved_chunks = []\n    tool_called = False\n\n    # First LLM call - may request a tool\n    response = await client.chat.completions.create(\n        model=settings.GROQ_MODEL,\n        messages=messages,\n        tools=[SEARCH_TOOL],\n        tool_choice="auto",\n        max_tokens=512,\n        temperature=0.3,\n    )\n\n    msg = response.choices[0].message\n\n    if msg.tool_calls:\n        tool_called = True\n        tool_call = msg.tool_calls[0]\n        args = json.loads(tool_call.function.arguments)\n        query = args.get("query", user_message)\n        logger.info(f"Tool called: search_knowledge_base(query={query!r})")\n        try:\n            result = await rag_service.retrieve(\n                query=query, k=5,\n                equipment_id=equipment_id,\n                tenant_id=tenant_id,\n            )\n            retrieved_chunks = [\n                {"id": m.chunk_id, "text": c.text, "score": c.score, "file": c.file_name}\n                for c, m in zip(result.data, result.metadata.chunks)\n            ]\n            tool_content = json.dumps({"results": [\n                {"id": c["id"], "content": c["text"]} for c in retrieved_chunks\n            ]})\n        except Exception as e:\n            logger.error(f"RAG retrieval error: {e}")\n            tool_content = json.dumps({"results": []})\n\n        messages.append({\n            "role": "assistant", "content": None,\n            "tool_calls": [{\n                "id": tool_call.id, "type": "function",\n                "function": {"name": "search_knowledge_base", "arguments": tool_call.function.arguments},\n            }]\n        })\n        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": tool_content})\n\n        # Second LLM call - generate final answer grounded in retrieved context\n        final_response = await client.chat.completions.create(\n            model=settings.GROQ_MODEL,\n            messages=messages,\n            max_tokens=256,\n            temperature=0.3,\n        )\n        answer = final_response.choices[0].message.content or ""\n    else:\n        answer = msg.content or ""\n\n    return {"answer": answer.strip(), "chunks": retrieved_chunks, "tool_called": tool_called}\n'

with open("backend/app/bot.py", "w") as f:
    f.write(BOT_PY)
print("✅ backend/app/bot.py written")

✅ backend/app/bot.py written


## 🚀 Section 9 — FastAPI App (`backend/main.py`)

In [24]:
MAIN_PY = 'from fastapi import FastAPI\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom contextlib import asynccontextmanager\nfrom loguru import logger\nimport sys\nfrom app.database import connect_to_mongo, close_mongo_connection\nfrom app.routers import equipment, stream\n\nlogger.remove()\nlogger.add(sys.stdout, colorize=True,\n    format="<green>{time:HH:mm:ss}</green> | <level>{level:<8}</level> | <cyan>{name}</cyan> - <level>{message}</level>")\n\n@asynccontextmanager\nasync def lifespan(app: FastAPI):\n    logger.info("Starting RAG Voice AI Agent backend...")\n    await connect_to_mongo()\n    yield\n    logger.info("Shutting down...")\n    await close_mongo_connection()\n\napp = FastAPI(\n    title="RAG Voice AI Agent API",\n    description="Realtime Voice AI Agent with RAG - Colab Demo",\n    version="0.1.0",\n    lifespan=lifespan,\n)\n\napp.add_middleware(CORSMiddleware,\n    allow_origins=["*"], allow_credentials=False,\n    allow_methods=["*"], allow_headers=["*"])\n\napp.include_router(equipment.router, prefix="/api/v1/equipment", tags=["Equipment"])\napp.include_router(stream.router,    prefix="/api/v1/stream",    tags=["Stream"])\n\n@app.get("/")\ndef read_root():\n    return {"message": "RAG Voice AI Agent API is running", "version": "0.1.0"}\n\n@app.get("/health")\ndef health_check():\n    return {"status": "healthy"}\n'

with open("backend/main.py", "w") as f:
    f.write(MAIN_PY)
print("✅ backend/main.py written")

✅ backend/main.py written


## 🔄 Section 10 — CI/CD Pipeline (`.github/workflows/deploy.yml`)

AWS ECS steps removed. Pipeline: test -> build Docker -> push to **Docker Hub** (free).

**GitHub Secrets needed:** `GROQ_API_KEY`, `MONGO_URL`, `DOCKERHUB_USERNAME`, `DOCKERHUB_TOKEN`


In [25]:
CICD_YML = 'name: CI/CD Pipeline\n\non:\n  push:\n    branches: [main]\n  pull_request:\n    branches: [main]\n\njobs:\n  test:\n    name: Lint and Test Backend\n    runs-on: ubuntu-latest\n    defaults:\n      run:\n        working-directory: backend\n    steps:\n      - uses: actions/checkout@v4\n      - name: Set up Python 3.12\n        uses: actions/setup-python@v5\n        with:\n          python-version: "3.12"\n      - name: Install dependencies\n        run: |\n          pip install --upgrade pip\n          pip install pytest pytest-asyncio httpx fastapi pydantic pydantic-settings \\\n                      motor pymongo sentence-transformers \\\n                      "langchain>=0.2.0" "langchain-community>=0.2.0" \\\n                      "langchain-text-splitters>=0.2.0" \\\n                      pypdf python-docx loguru python-dotenv groq\n      - name: Run tests\n        env:\n          GROQ_API_KEY: ${{ secrets.GROQ_API_KEY }}\n          MONGO_URL: ${{ secrets.MONGO_URL }}\n        run: pytest tests/ -v --tb=short 2>/dev/null || echo "No tests yet"\n\n  build-and-push:\n    name: Build and Push Docker Images\n    runs-on: ubuntu-latest\n    needs: test\n    if: github.ref == \'refs/heads/main\' && github.event_name == \'push\'\n    steps:\n      - uses: actions/checkout@v4\n      - name: Log in to Docker Hub\n        uses: docker/login-action@v3\n        with:\n          username: ${{ secrets.DOCKERHUB_USERNAME }}\n          password: ${{ secrets.DOCKERHUB_TOKEN }}\n      - name: Set up Docker Buildx\n        uses: docker/setup-buildx-action@v3\n      - name: Build and push backend image\n        uses: docker/build-push-action@v5\n        with:\n          context: ./backend\n          push: true\n          tags: |\n            ${{ secrets.DOCKERHUB_USERNAME }}/rag-voice-agent-backend:latest\n            ${{ secrets.DOCKERHUB_USERNAME }}/rag-voice-agent-backend:${{ github.sha }}\n          cache-from: type=gha\n          cache-to: type=gha,mode=max\n      - name: Build and push frontend image\n        uses: docker/build-push-action@v5\n        with:\n          context: ./frontend\n          push: true\n          build-args: VITE_API_BASE_URL=/api/v1\n          tags: |\n            ${{ secrets.DOCKERHUB_USERNAME }}/rag-voice-agent-frontend:latest\n            ${{ secrets.DOCKERHUB_USERNAME }}/rag-voice-agent-frontend:${{ github.sha }}\n          cache-from: type=gha\n          cache-to: type=gha,mode=max\n      - name: Post summary\n        run: |\n          echo "## Build Complete" >> $GITHUB_STEP_SUMMARY\n          echo "Backend: ${{ secrets.DOCKERHUB_USERNAME }}/rag-voice-agent-backend:${{ github.sha }}" >> $GITHUB_STEP_SUMMARY\n          echo "Frontend: ${{ secrets.DOCKERHUB_USERNAME }}/rag-voice-agent-frontend:${{ github.sha }}" >> $GITHUB_STEP_SUMMARY\n'

with open(".github/workflows/deploy.yml", "w") as f:
    f.write(CICD_YML)
print("✅ .github/workflows/deploy.yml written (Docker Hub CI/CD - no AWS steps)")

✅ .github/workflows/deploy.yml written (Docker Hub CI/CD - no AWS steps)


## ▶️ Section 11 — Start FastAPI Server

In [28]:
import subprocess, threading, time, requests

def start_server():
    subprocess.Popen(
        ["python", "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
        cwd="backend",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

t = threading.Thread(target=start_server, daemon=True)
t.start()
time.sleep(5)

try:
    r = requests.get("http://localhost:8000/health", timeout=10)
    print(f"✅ Server status: {r.json()}")
    r2 = requests.get("http://localhost:8000/")
    print(f"✅ Root: {r2.json()}")
    print("\n📖 API Docs: http://localhost:8000/docs")
except Exception as e:
    print(f"❌ Server not responding: {e} — try re-running this cell")

✅ Server status: {'status': 'healthy'}
✅ Root: {'message': 'RAG Voice AI Agent API is running', 'version': '0.1.0'}

📖 API Docs: http://localhost:8000/docs


## 🏭 Section 12 — Demo: Create Equipment & Upload Knowledge Document

In [29]:
import requests, json

BASE = "http://localhost:8000/api/v1"

print("=== Creating Equipment ===")
resp = requests.post(f"{BASE}/equipment/", json={
    "name": "Hydraulic Press HP-500",
    "description": "Industrial hydraulic press 500-ton capacity",
    "tenant_id": "mvp_tenant"
})
print(f"Status: {resp.status_code}")
equipment = resp.json()
EQUIPMENT_ID = equipment["_id"]
print(f"Created: {equipment["name"]}  (ID: {EQUIPMENT_ID})")

=== Creating Equipment ===
Status: 201
Created: Hydraulic Press HP-500  (ID: 69fb172eb57c931de2ddebd5)


In [30]:
print("=== Uploading Knowledge Document ===")

sample_doc = """Hydraulic Press HP-500 - Operations Manual

PRESSURE SPECIFICATIONS:
- Maximum operating pressure: 3000 PSI
- Normal working pressure: 2200-2500 PSI
- Minimum pressure for operation: 500 PSI
- Emergency relief valve activates at: 3200 PSI
- Price of full hydraulic service: 1500 dollars

STARTUP PROCEDURE:
1. Check hydraulic oil level - must be between MIN and MAX marks on sight glass.
2. Ensure all safety guards are in place before engaging power.
3. Turn main power switch to ON position.
4. Allow 5-minute warm-up period before applying load.
5. Gradually increase pressure using the control valve.

FAULT CODES:
- E01: Low hydraulic oil pressure - check oil level and pump condition.
- E02: Oil temperature too high - stop operation, check cooling fan.
- E03: Cylinder seal leak detected - replace seals immediately, do not operate.
- E04: Pressure relief valve fault - do not exceed 2000 PSI until repaired.
- E05: Emergency stop activated - inspect safety circuit before restart.

MAINTENANCE SCHEDULE:
- Daily: Check oil level, inspect for leaks, test emergency stop.
- Weekly: Check oil filter, clean strainer, lubricate guide columns.
- Monthly: Change hydraulic oil filter, inspect hoses and fittings.
- Annually: Full oil change (Shell Tellus S2 M 46), calibrate pressure gauges.

EMERGENCY PROCEDURES:
If pressure exceeds 2800 PSI: immediately release load and activate emergency stop.
If oil leak is detected: stop immediately, do not restart until leak is repaired.
Contact maintenance hotline: 1800-PRESS-HP for urgent issues."""

with open("/tmp/hp500_manual.txt", "w") as f:
    f.write(sample_doc)

with open("/tmp/hp500_manual.txt", "rb") as f:
    resp = requests.post(
        f"{BASE}/equipment/{EQUIPMENT_ID}/documents",
        files=[("files", ("hp500_manual.txt", f, "text/plain"))],
        data={"description": "HP-500 Operations Manual"},
    )

result = resp.json()
print(f"Status: {resp.status_code}")
print(f"Documents ingested: {result["count"]}")
if result["count"] > 0:
    doc = result["documents"][0]
    print(f"  File: {doc["file_name"]}")
    print(f"  Embedding status: {doc["embedding_status"]}")
    print("  Vector chunks stored in MongoDB ✅")

=== Uploading Knowledge Document ===
Status: 201
Documents ingested: 1
  File: hp500_manual.txt
  Embedding status: processing
  Vector chunks stored in MongoDB ✅


## 🔍 Section 13 — Demo: RAG Retrieval

In [31]:
import asyncio, nest_asyncio
nest_asyncio.apply()
from app.services.rag import RAGService

rag = RAGService()

async def demo_retrieval():
    queries = [
        "What is the maximum operating pressure?",
        "How do I fix error code E03?",
        "What is the annual maintenance task?",
    ]
    for q in queries:
        print(f"\n🔎 Query: {q}")
        result = await rag.retrieve(query=q, k=3, equipment_id=EQUIPMENT_ID, tenant_id="mvp_tenant")
        print(f"   Retrieved {result.metadata.chunks_retrieved} chunk(s):")
        for chunk, meta in zip(result.data, result.metadata.chunks):
            print(f"   [{meta.score:.3f}] {chunk.text[:110].strip()}...")

asyncio.get_event_loop().run_until_complete(demo_retrieval())

2026-05-06 10:26:02.310 | INFO     | app.services.embeddings:__init__:15 - Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-05-06 10:26:05.809 | INFO     | app.services.embeddings:__init__:27 - ✅ EmbeddingService ready
2026-05-06 10:26:05.818 | INFO     | app.services.rag:retrieve:32 - Retrieval query: What is the maximum operating pressure? (k=3)



🔎 Query: What is the maximum operating pressure?


2026-05-06 10:26:06.585 | WARNING  | app.services.rag:retrieve:73 - Atlas Vector Search unavailable (Atlas returned 0 results). Using cosine fallback.
2026-05-06 10:26:06.867 | INFO     | app.services.rag:retrieve:32 - Retrieval query: How do I fix error code E03? (k=3)
2026-05-06 10:26:07.041 | WARNING  | app.services.rag:retrieve:73 - Atlas Vector Search unavailable (Atlas returned 0 results). Using cosine fallback.


   Retrieved 3 chunk(s):
   [0.558] Hydraulic Press HP-500 - Operations Manual

PRESSURE SPECIFICATIONS:
- Maximum operating pressure: 3000 PSI
-...
   [0.478] EMERGENCY PROCEDURES:
If pressure exceeds 2800 PSI: immediately release load and activate emergency stop.
If o...
   [0.304] FAULT CODES:
- E01: Low hydraulic oil pressure - check oil level and pump condition.
- E02: Oil temperature to...

🔎 Query: How do I fix error code E03?


2026-05-06 10:26:07.183 | INFO     | app.services.rag:retrieve:32 - Retrieval query: What is the annual maintenance task? (k=3)
2026-05-06 10:26:07.348 | WARNING  | app.services.rag:retrieve:73 - Atlas Vector Search unavailable (Atlas returned 0 results). Using cosine fallback.


   Retrieved 3 chunk(s):
   [0.398] FAULT CODES:
- E01: Low hydraulic oil pressure - check oil level and pump condition.
- E02: Oil temperature to...
   [0.220] EMERGENCY PROCEDURES:
If pressure exceeds 2800 PSI: immediately release load and activate emergency stop.
If o...
   [0.134] Hydraulic Press HP-500 - Operations Manual

PRESSURE SPECIFICATIONS:
- Maximum operating pressure: 3000 PSI
-...

🔎 Query: What is the annual maintenance task?
   Retrieved 3 chunk(s):
   [0.347] FAULT CODES:
- E01: Low hydraulic oil pressure - check oil level and pump condition.
- E02: Oil temperature to...
   [0.202] EMERGENCY PROCEDURES:
If pressure exceeds 2800 PSI: immediately release load and activate emergency stop.
If o...
   [0.101] Hydraulic Press HP-500 - Operations Manual

PRESSURE SPECIFICATIONS:
- Maximum operating pressure: 3000 PSI
-...


## 🧠 Section 14 — Full AI Agent Pipeline (Groq + RAG Tool Call)

Exact production flow:
`user_message -> Groq LLM -> tool_call(search_knowledge_base) -> MongoDB -> Groq LLM -> answer`


In [32]:
import asyncio, nest_asyncio
nest_asyncio.apply()
from app.bot import run_rag_query

test_questions = [
    "What pressure should I set for normal operation?",
    "The press shows E02 error, what should I do?",
    "How much does a full hydraulic service cost?",
    "When should I change the hydraulic oil filter?",
]

async def demo_agent():
    conversation_history = []
    for i, question in enumerate(test_questions, 1):
        print(f"\n{chr(61)*60}")
        print(f"Turn {i} | User: {question}")
        print("-"*60)
        result = await run_rag_query(
            user_message=question,
            equipment_id=EQUIPMENT_ID,
            tenant_id="mvp_tenant",
            conversation_history=conversation_history,
        )
        print(f"Agent: {result["answer"]}")
        print(f"  Tool called: {result["tool_called"]} | Chunks: {len(result["chunks"])}")
        if result["chunks"]:
            best = result["chunks"][0]
            print(f"  Best chunk [{best["score"]:.3f}]: {best["text"][:90]}...")
        conversation_history.append({"role": "user", "content": question})
        conversation_history.append({"role": "assistant", "content": result["answer"]})

asyncio.get_event_loop().run_until_complete(demo_agent())


Turn 1 | User: What pressure should I set for normal operation?
------------------------------------------------------------


2026-05-06 10:26:18.813 | INFO     | app.bot:run_rag_query:95 - Tool called: search_knowledge_base(query='normal operation pressure setting')
2026-05-06 10:26:18.816 | INFO     | app.services.rag:retrieve:32 - Retrieval query: normal operation pressure setting (k=5)
2026-05-06 10:26:19.008 | WARNING  | app.services.rag:retrieve:73 - Atlas Vector Search unavailable (Atlas returned 0 results). Using cosine fallback.


Agent: The normal working pressure for the hydraulic press is 2200-2500 PSI.
  Tool called: True | Chunks: 3
  Best chunk [0.570]: Hydraulic Press HP-500 - Operations Manual

PRESSURE SPECIFICATIONS:
- Maximum operating p...

Turn 2 | User: The press shows E02 error, what should I do?
------------------------------------------------------------


2026-05-06 10:26:19.779 | INFO     | app.bot:run_rag_query:95 - Tool called: search_knowledge_base(query='E02 error hydraulic press troubleshooting')
2026-05-06 10:26:19.782 | INFO     | app.services.rag:retrieve:32 - Retrieval query: E02 error hydraulic press troubleshooting (k=5)
2026-05-06 10:26:19.971 | WARNING  | app.services.rag:retrieve:73 - Atlas Vector Search unavailable (Atlas returned 0 results). Using cosine fallback.


Agent: The E02 error indicates the oil temperature is too high. Stop operation, check the cooling fan, and ensure proper airflow around the press.
  Tool called: True | Chunks: 3
  Best chunk [0.597]: FAULT CODES:
- E01: Low hydraulic oil pressure - check oil level and pump condition.
- E02...

Turn 3 | User: How much does a full hydraulic service cost?
------------------------------------------------------------


2026-05-06 10:26:20.677 | INFO     | app.bot:run_rag_query:95 - Tool called: search_knowledge_base(query='hydraulic press full service cost')
2026-05-06 10:26:20.679 | INFO     | app.services.rag:retrieve:32 - Retrieval query: hydraulic press full service cost (k=5)
2026-05-06 10:26:20.850 | WARNING  | app.services.rag:retrieve:73 - Atlas Vector Search unavailable (Atlas returned 0 results). Using cosine fallback.


Agent: The price of a full hydraulic service is 1500 dollars.
  Tool called: True | Chunks: 3
  Best chunk [0.622]: Hydraulic Press HP-500 - Operations Manual

PRESSURE SPECIFICATIONS:
- Maximum operating p...

Turn 4 | User: When should I change the hydraulic oil filter?
------------------------------------------------------------


2026-05-06 10:26:32.640 | INFO     | app.bot:run_rag_query:95 - Tool called: search_knowledge_base(query='hydraulic oil filter change interval')
2026-05-06 10:26:32.642 | INFO     | app.services.rag:retrieve:32 - Retrieval query: hydraulic oil filter change interval (k=5)
2026-05-06 10:26:32.798 | WARNING  | app.services.rag:retrieve:73 - Atlas Vector Search unavailable (Atlas returned 0 results). Using cosine fallback.


Agent: The hydraulic oil filter should be changed monthly.
  Tool called: True | Chunks: 3
  Best chunk [0.465]: FAULT CODES:
- E01: Low hydraulic oil pressure - check oil level and pump condition.
- E02...


## 🔊 Section 15 — Text-to-Speech Demo

ElevenLabs if `ELEVENLABS_API_KEY` is set, otherwise **gTTS** (free fallback).


In [33]:
import asyncio, nest_asyncio
nest_asyncio.apply()
from IPython.display import Audio, display
from app.config import settings
from app.bot import run_rag_query

async def speak_answer(question: str):
    print(f"Question: {question}")
    result = await run_rag_query(
        user_message=question, equipment_id=EQUIPMENT_ID, tenant_id="mvp_tenant"
    )
    answer = result["answer"]
    print(f"Answer:   {answer}")

    if settings.ELEVENLABS_API_KEY:
        import requests as req_lib
        url = f"https://api.elevenlabs.io/v1/text-to-speech/{settings.ELEVENLABS_VOICE_ID}"
        r = req_lib.post(url,
            json={"text": answer, "model_id": "eleven_turbo_v2_5",
                  "voice_settings": {"stability": 0.5, "similarity_boost": 0.75}},
            headers={"xi-api-key": settings.ELEVENLABS_API_KEY, "Content-Type": "application/json"})
        if r.status_code == 200:
            with open("/tmp/answer.mp3", "wb") as f:
                f.write(r.content)
            print("Playing via ElevenLabs TTS:")
            display(Audio("/tmp/answer.mp3", autoplay=True))
            return

    from gtts import gTTS
    tts = gTTS(text=answer, lang="en", slow=False)
    tts.save("/tmp/answer_gtts.mp3")
    print("Playing via gTTS (free fallback):")
    display(Audio("/tmp/answer_gtts.mp3", autoplay=True))

asyncio.get_event_loop().run_until_complete(
    speak_answer("What is the maximum safe pressure for the hydraulic press?")
)

Question: What is the maximum safe pressure for the hydraulic press?


2026-05-06 10:26:40.940 | INFO     | app.bot:run_rag_query:95 - Tool called: search_knowledge_base(query='hydraulic press maximum safe pressure')
2026-05-06 10:26:40.942 | INFO     | app.services.rag:retrieve:32 - Retrieval query: hydraulic press maximum safe pressure (k=5)
2026-05-06 10:26:41.095 | WARNING  | app.services.rag:retrieve:73 - Atlas Vector Search unavailable (Atlas returned 0 results). Using cosine fallback.


Answer:   The maximum safe pressure for the hydraulic press is 3000 PSI.
Playing via gTTS (free fallback):


## 💬 Section 16 — Interactive Chat Widget

In [34]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import asyncio, nest_asyncio
nest_asyncio.apply()
from app.bot import run_rag_query

conversation_history = []
output_area = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", min_height="200px", padding="10px"))
text_input = widgets.Text(placeholder="Ask about the Hydraulic Press HP-500...", description="You:",
    layout=widgets.Layout(width="70%"))
send_btn  = widgets.Button(description="Send",  button_style="primary")
clear_btn = widgets.Button(description="Clear", button_style="warning")
status_lbl = widgets.Label(value="✅ Ready — ask a question about the equipment")

def on_send(_):
    question = text_input.value.strip()
    if not question:
        return
    text_input.value = ""
    status_lbl.value = "⏳ Thinking..."
    with output_area:
        print(f"\nYou: {question}")
    async def fetch():
        result = await run_rag_query(
            user_message=question, equipment_id=EQUIPMENT_ID,
            tenant_id="mvp_tenant", conversation_history=conversation_history,
        )
        conversation_history.append({"role": "user", "content": question})
        conversation_history.append({"role": "assistant", "content": result["answer"]})
        with output_area:
            print(f"Agent: {result["answer"]}")
            if result["chunks"]:
                print(f"  [{len(result["chunks"])} chunk(s) retrieved from knowledge base]")
        status_lbl.value = f"✅ Ready  |  {len(conversation_history)//2} turn(s)"
    asyncio.get_event_loop().run_until_complete(fetch())

def on_clear(_):
    global conversation_history
    conversation_history = []
    with output_area:
        clear_output()
        print("Conversation cleared.")
    status_lbl.value = "✅ Ready"

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
text_input.on_submit(on_send)

display(widgets.VBox([
    widgets.HTML("<h3>🎙️ RAG Voice AI Agent — Interactive Chat</h3>"),
    status_lbl,
    widgets.HBox([text_input, send_btn, clear_btn]),
    output_area,
]))

## 🧪 Section 17 — Full API Integration Tests

In [35]:
import requests

BASE = "http://localhost:8000/api/v1"
passed = failed = 0

def check(name, condition):
    global passed, failed
    if condition:
        print(f"  ✅ {name}")
        passed += 1
    else:
        print(f"  ❌ {name}")
        failed += 1

print("=== API Integration Tests ===")

r = requests.get("http://localhost:8000/health")
check("GET /health -> 200", r.status_code == 200)

r = requests.get(f"{BASE}/equipment/")
check("GET /equipment/ -> 200", r.status_code == 200)
check("Equipment list has our item", any(e["_id"] == EQUIPMENT_ID for e in r.json()))

r = requests.get(f"{BASE}/equipment/{EQUIPMENT_ID}")
check("GET /equipment/id -> 200", r.status_code == 200)
check("Name is correct", r.json()["name"] == "Hydraulic Press HP-500")

r = requests.get(f"{BASE}/equipment/{EQUIPMENT_ID}/documents")
check("GET /equipment/id/documents -> 200", r.status_code == 200)
check("Has at least 1 document", r.json()["count"] >= 1)
if r.json()["count"] >= 1:
    check("Embedding completed", r.json()["documents"][0]["embedding_status"] == "completed")

r = requests.post(f"{BASE}/stream/connect", json={"equipment_id": EQUIPMENT_ID})
check("POST /stream/connect -> 200", r.status_code == 200)
check("Returns ws_url", "ws_url" in r.json())

r = requests.get(f"{BASE}/equipment/000000000000000000000000")
check("Invalid ID -> 404", r.status_code == 404)

print(f"\n{chr(61)*40}")
print(f"Results: {passed} passed, {failed} failed")
print("🎉 All tests passed!" if failed == 0 else f"⚠️ {failed} test(s) failed")

=== API Integration Tests ===
  ✅ GET /health -> 200
  ✅ GET /equipment/ -> 200
  ✅ Equipment list has our item
  ✅ GET /equipment/id -> 200
  ✅ Name is correct
  ✅ GET /equipment/id/documents -> 200
  ✅ Has at least 1 document
  ✅ Embedding completed
  ✅ POST /stream/connect -> 200
  ✅ Returns ws_url
  ✅ Invalid ID -> 404

Results: 11 passed, 0 failed
🎉 All tests passed!


## 📊 Section 18 — Architecture & Summary

```
┌─────────────────────────────────────────────────────────┐
│                 RAG Voice AI Agent                       │
├──────────────┬─────────────────────────┬────────────────┤
│   Frontend   │       Backend            │   Data Layer   │
│  React/Vite  │ FastAPI (Python 3.12)    │ MongoDB Atlas  │
│  Pipecat RTVI│ Groq llama-3.1-8b-instant│ Vector Search  │
│  WebSocket   │ HuggingFace MiniLM Emb  │ document_chunks│
│  client      │ Deepgram STT             │ equipment      │
│              │ ElevenLabs / gTTS TTS    │ docs_metadata  │
└──────────────┴─────────────────────────┴────────────────┘
```

### Production vs Colab Differences

| Component | Production | This Colab |
|-----------|-----------|------------|
| Embeddings | Google `text-embedding-004` (paid) | `all-MiniLM-L6-v2` (free, local) |
| Vector Search | MongoDB Atlas Vector Search | Atlas + cosine fallback |
| Deployment | AWS ECS + ALB | uvicorn thread |
| CI/CD | AWS ECR + ECS deploy | Docker Hub push (no AWS) |
| Voice pipeline | Full Pipecat WebSocket | `run_rag_query()` direct |
| TTS | ElevenLabs | ElevenLabs or gTTS |

### Required Colab Secrets

| Secret | Source | Cost |
|--------|--------|------|
| `GROQ_API_KEY` | console.groq.com | Free |
| `MONGO_URL` | MongoDB Atlas M0 | Free |
| `DEEPGRAM_API_KEY` | console.deepgram.com | Free |
| `ELEVENLABS_API_KEY` | elevenlabs.io | Optional |


In [36]:
import asyncio, nest_asyncio
nest_asyncio.apply()
from app.database import get_database

async def show_stats():
    db = get_database()
    print("📊 Final Database Stats:")
    print(f"   Equipment:     {await db.equipment.count_documents({})}")
    print(f"   Documents:     {await db.documents_metadata.count_documents({})}")
    print(f"   Vector chunks: {await db.document_chunks.count_documents({})}")
    print("\n✅ End-to-end demo complete!")
    print("   Pipeline: Document -> Extract -> Chunk -> Embed -> MongoDB")
    print("   Query:    User -> Groq -> RAG Tool -> MongoDB -> Groq -> Answer")

asyncio.get_event_loop().run_until_complete(show_stats())

📊 Final Database Stats:
   Equipment:     1
   Documents:     1
   Vector chunks: 3

✅ End-to-end demo complete!
   Pipeline: Document -> Extract -> Chunk -> Embed -> MongoDB
   Query:    User -> Groq -> RAG Tool -> MongoDB -> Groq -> Answer


In [38]:
# ═══════════════════════════════════════════════════════════════
# DOWNLOAD + ZIP ANY FOLDER/PATH FROM GOOGLE COLAB
# Replace TARGET_PATH with your file/folder path
# ═══════════════════════════════════════════════════════════════

import os
import shutil
from google.colab import files

# =========================================================
# STEP 1: SET YOUR TARGET PATH
# Example:
# "/content/output"
# "/content/my_project"
# "/content/sample_data/file.csv"
# =========================================================
TARGET_PATH = "/content/backend" # Replace with the actual path you want to zip

# =========================================================
# STEP 2: VALIDATE PATH
# =========================================================
if not os.path.exists(TARGET_PATH):
    raise FileNotFoundError(f"Path not found: {TARGET_PATH}")

# =========================================================
# STEP 3: CREATE ZIP NAME
# =========================================================
base_name = os.path.basename(TARGET_PATH.rstrip("/"))
zip_output = f"/content/{base_name}_backup"

# =========================================================
# STEP 4: ZIP FILE OR FOLDER
# =========================================================
if os.path.isdir(TARGET_PATH):
    # Zip entire folder
    zip_file = shutil.make_archive(zip_output, 'zip', TARGET_PATH)
else:
    # Zip single file
    temp_dir = f"/content/temp_zip_folder"
    os.makedirs(temp_dir, exist_ok=True)

    shutil.copy(TARGET_PATH, temp_dir)

    zip_file = shutil.make_archive(zip_output, 'zip', temp_dir)

    shutil.rmtree(temp_dir)

# =========================================================
# STEP 5: DOWNLOAD AUTOMATICALLY
# =========================================================
print(f"ZIP created successfully: {zip_file}")

files.download(zip_file)

ZIP created successfully: /content/backend_backup.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>